In [1]:
import jax
# Verify that JAX sees the Google TPU accelerator
print("Available devices:", jax.devices())

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Available devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


In [2]:
import jax.numpy as jnp
import jax.random as jrandom
import time

# 1. Define the simulation as a pure mathematical function
def tpu_monte_carlo(seed, S0, K, r, sigma, T, num_paths):
    # Generate a massive vector of random normal variables on the TPU hardware
    key = jrandom.PRNGKey(seed)
    Z = jrandom.normal(key, shape=(num_paths,))

    # Vectorized Geometric Brownian Motion equation
    ST = S0 * jnp.exp((r - 0.5 * sigma**2) * T + sigma * jnp.sqrt(T) * Z)

    # Element-wise option payoff evaluation
    payoffs = jnp.maximum(ST - K, 0.0)

    # TPU hardware reduction (Parallel Average) and discounting
    price = jnp.mean(payoffs) * jnp.exp(-r * T)
    return price

# 2. Compile the function using XLA (Accelerated Linear Algebra) for the TPU
jit_tpu_monte_carlo = jax.jit(tpu_monte_carlo, static_argnums=(6,))

# 3. Parameters
num_paths = 10_000_000
S0, K, r, sigma, T = 100.0, 105.0, 0.05, 0.2, 1.0

# Warm-up run (forces the XLA compiler to build the TPU hardware kernel)
_ = jit_tpu_monte_carlo(42, S0, K, r, sigma, T, num_paths)

# 4. Benchmark the compiled TPU execution
start_tpu = time.time()
tpu_price = jit_tpu_monte_carlo(42, S0, K, r, sigma, T, num_paths)
# Block until calculation finishes to get an accurate time measurement
tpu_price.block_until_ready()
tpu_time = time.time() - start_tpu

print(f"=== TPU v5e PERFORMANCE PROFILE ===")
print(f"Calculated Option Price: ${float(tpu_price):.4f}")
print(f"TPU Execution Time (10M Paths): {tpu_time:.6f} seconds")

=== TPU v5e PERFORMANCE PROFILE ===
Calculated Option Price: $8.0253
TPU Execution Time (10M Paths): 0.000805 seconds
